# N101 · 0/1背包与逆序容量

**目标：** 证明滚动数组逆序更新避免重复用物品。

**先修：** N098, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 物品维度；容量；最大值/可达；恰好/至多；负无穷初始化。

## 从问题到算法

0/1背包的第i层只允许前i件物品。压成一维时必须逆序容量：读取dp[c−w]时它仍来自上一层，因此同一物品不会重复使用。符号和计数还必须区分“最优值”与“方案数”；值为0的元素依然有正负两种选择。

## 最小实现

**本章接口：** `knapsack_01(weights, values, capacity)`、`can_partition(nums)`、`find_target_sum_ways(nums, target)`

In [1]:
from collections import Counter

def knapsack_01(weights,values,capacity):
    if len(weights)!=len(values) or capacity<0 or any(w<0 for w in weights): raise ValueError('invalid knapsack')
    dp=[0]*(capacity+1)
    for w,v in zip(weights,values):
        for c in range(capacity,w-1,-1): dp[c]=max(dp[c],dp[c-w]+v)
    return dp[-1]

def can_partition(nums):
    total=sum(nums)
    if any(x<0 for x in nums): raise ValueError('nonnegative values required')
    if total%2: return False
    reachable=1
    for x in nums: reachable|=reachable<<x
    return bool(reachable>>(total//2)&1)

def find_target_sum_ways(nums,target):
    states=Counter({0:1})
    for x in nums:
        nxt=Counter()
        for value,count in states.items(): nxt[value+x]+=count; nxt[value-x]+=count
        states=nxt
    return states[target]

## 正确性、前提与复杂度

背包按“取第i件”与“不取”划分；逆序保证转移读取旧层。位集的第s位代表和s可达，左移x对应加入物品x；或操作合并两种可能。计数Counter每一步创建新层，零值对应两条不同决策边，所以自然翻倍。

**代价：** 背包O(nC)时间O(C)空间；位集在总和S位上做n次移位/或，位复杂度O(nS)上界、O(S)位空间；符号计数至多O(nS)状态处理，S=各数绝对值之和。这些是伪多项式代价，不是对输入比特长度的多项式。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

weights=[2,3,4]; values=[4,5,7]; cap=6; dp=[0]*(cap+1); rows=[('无物品',dp[:])]
for w,v in zip(weights,values):
    for c in range(cap,w-1,-1): dp[c]=max(dp[c],dp[c-w]+v)
    rows.append((f'w={w},v={v}',dp[:]))
show_table(['已处理物品','各容量最优值'],rows)

已处理物品,各容量最优值
无物品,"[0, 0, 0, 0, 0, 0, 0]"
"w=2,v=4","[0, 0, 4, 4, 4, 4, 4]"
"w=3,v=5","[0, 0, 4, 5, 5, 9, 9]"
"w=4,v=7","[0, 0, 4, 5, 7, 9, 11]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert can_partition([1,5,11,5])
assert knapsack_01([2],[3],4)==3
assert find_target_sum_ways([0,0],0)==4
from itertools import product
from random import Random
rng=Random(101)
for _ in range(140):
    n=rng.randrange(7); w=[rng.randrange(5) for _ in range(n)]; v=[rng.randrange(-2,9) for _ in range(n)]; cap=rng.randrange(9)
    want=max(sum(v[i] for i in range(n) if mask>>i&1) for mask in range(1<<n) if sum(w[i] for i in range(n) if mask>>i&1)<=cap)
    assert knapsack_01(w,v,cap)==want
    assert can_partition(w)==any(2*sum(w[i] for i in range(n) if mask>>i&1)==sum(w) for mask in range(1<<n))
for a in product(range(3),repeat=5):
    for target in range(-2,3):
        assert find_target_sum_ways(a,target)==sum(sum(s*x for s,x in zip(signs,a))==target for signs in product((-1,1),repeat=5))
print('N101: 所有本章断言通过')

N101: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用正序更新构造重复拿物品反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 推导目标和向子集计数的转换。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 416. Partition Equal Subset Sum（canonical）
- 494. Target Sum（canonical）
- 1049. Last Stone Weight II（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。